In [ ]:
from project_utils import load_table, input_path, output_path, calculateCI, mean_error, standard_error, percentile_column, positive_fraction, SEED
import numpy as np
np.random.seed(SEED)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import scipy


In [ ]:
# calculate CI using bootstrap


In [ ]:
f1 = load_table('citing_papers')
f2 = load_table('nobel_papers')


In [ ]:
f1['Nobelyear'] = f1['Nobel'].map(f2.set_index('paperid')['paperyear'])
f1['gap'] = f1['paperyear'] - f1['Nobelyear']
f1['type'] = f1['type'].map({'CC':'CCP','DC':'DCP'})


In [ ]:
f1['Field'] = f1['Nobel'].map(f2.set_index('paperid')['Field'])
f11 = f1[f1['Field'] == 'Physics']
f12 = f1[f1['Field'] == 'Chemistry']
f13 = f1[f1['Field'] == 'Medicine']


# FIG 3


In [ ]:
f11D = f11[f11['type']=='DCP']
f12D = f12[f12['type']=='DCP']
f13D = f13[f13['type']=='DCP']
f11C = f11[f11['type']=='CCP']
f12C = f12[f12['type']=='CCP']
f13C = f13[f13['type']=='CCP']


In [ ]:
def getmeanCI(ff):
    means, errors, samples = [], [], []
    for gap in range(30):
        counts = ff.loc[ff['gap'].eq(gap)].groupby('Nobel')['paperid'].count().to_numpy()
        means.append(counts.mean() if len(counts) else np.nan)
        errors.append(mean_error(counts))
        samples.append(counts)
    return samples, np.asarray(means), np.asarray(errors).T


In [ ]:
D1, meanlst1D,CIlst1D = getmeanCI(f11D)
D2, meanlst2D,CIlst2D = getmeanCI(f12D)
D3, meanlst3D,CIlst3D = getmeanCI(f13D)
C1, meanlst1C,CIlst1C = getmeanCI(f11C)
C2, meanlst2C,CIlst2C = getmeanCI(f12C)
C3, meanlst3C,CIlst3C = getmeanCI(f13C)


In [ ]:
meanD = [meanlst1D,meanlst2D,meanlst3D]
meanC = [meanlst1C,meanlst2C,meanlst3C]

CID = [CIlst1D,CIlst2D,CIlst3D]
CIC = [CIlst1C,CIlst2C,CIlst3C]

dD = [D1,D2,D3]
dC = [C1,C2,C3]


In [ ]:
clst = ["#154360","#E67E22"]
glst = [18.2,16,16.7]


In [ ]:
plt.subplots(3,1,figsize=(18,15),sharex=True,sharey=True)
plt.subplots_adjust(left=None,bottom=None,right=None,top=None,wspace=0,hspace=0.08)
for _ in range(3):
    ax = plt.subplot(3,1,_ + 1)
    plt.bar(np.array(range(30))-.08,meanC[_],edgecolor = clst[1], yerr=CIC[_],align='center',capsize=3,linewidth = 4,fill = False,alpha = 0.7,width=0.6,label = 'CCP')
    plt.bar(np.array(range(30))+.08,meanD[_],edgecolor = clst[0], yerr=CID[_],align='center',capsize=3,linewidth = 4,fill = False,alpha = 1,width=0.6,label = 'DCP')
    plt.xlim(-1,30)
    plt.ylim(1,1000)
    ax.set_yscale("log")
    plt.yticks([10e0,10e1,10e2],size = 28)
    plt.xticks(size = 28)
    plt.legend(fontsize = 22,loc='upper right')
    plt.ylabel('',fontsize = 0)
    plt.xlabel('',fontsize = 0)
    for i in range(30):
        if len(dD[_][i]) == 0 or len(dC[_][i]) == 0:
            continue
        p = scipy.stats.mannwhitneyu(dD[_][i] , dC[_][i], alternative='two-sided').pvalue
        if not np.isfinite(p):
            continue
        if p>0.1:
            plt.text(i , meanC[_][i]+50, 'N', fontsize=15, ha='center', fontweight='bold')
        elif p>0.05:
            plt.text(i , meanC[_][i]+50, '*', fontsize=15, ha='center', fontweight='bold')
        elif p>0.01:
            plt.text(i , meanC[_][i]+50, '**', fontsize=15, ha='center', fontweight='bold')
        else:
            plt.text(i , meanC[_][i]+50, '***', fontsize=15, ha='center', fontweight='bold')
    
    
    
plt.savefig(output_path('fig_3_DCP_CCP_12.pdf'),bbox_inches = 'tight')


In [ ]:
plt.subplots(3,1,figsize=(18,15),sharex=True,sharey=True)
plt.subplots_adjust(left=None,bottom=None,right=None,top=None,wspace=0,hspace=0.08)
for _ in range(3):
    ax = plt.subplot(3,1,_ + 1)
    plt.bar(np.array(range(30))-.08,meanC[_],edgecolor = clst[1], yerr=CIC[_],align='center',capsize=3,linewidth = 4,fill = False,alpha = 0.7,width=0.6,label = 'CCP')
    plt.bar(np.array(range(30))+.08,meanD[_],edgecolor = clst[0], yerr=CID[_],align='center',capsize=3,linewidth = 4,fill = False,alpha = 1,width=0.6,label = 'DCP')
    plt.xlim(-1,30)
    plt.ylim(1,1000)
    ax.set_yscale("log")
    plt.yticks([10e0,10e1,10e2],size = 28)
    plt.xticks(size = 28)
    plt.legend(fontsize = 22,loc='upper right')
    plt.ylabel('',fontsize = 0)
    plt.xlabel('',fontsize = 0)
    for i in range(30):
        if len(dD[_][i]) == 0 or len(dC[_][i]) == 0:
            continue
        p = scipy.stats.ks_2samp(dD[_][i] , dC[_][i], alternative='two-sided').pvalue
        if not np.isfinite(p):
            continue
        if p>0.1:
            plt.text(i , meanC[_][i]+50, 'N', fontsize=15, ha='center', fontweight='bold')
        elif p>0.05:
            plt.text(i , meanC[_][i]+50, '*', fontsize=15, ha='center', fontweight='bold')
        elif p>0.01:
            plt.text(i , meanC[_][i]+50, '**', fontsize=15, ha='center', fontweight='bold')
        else:
            plt.text(i , meanC[_][i]+50, '***', fontsize=15, ha='center', fontweight='bold')


In [ ]:
plt.subplots(3,1,figsize=(18,15),sharex=True,sharey=True)
plt.subplots_adjust(left=None,bottom=None,right=None,top=None,wspace=0,hspace=0.08)
for _ in range(3):
    ax = plt.subplot(3,1,_ + 1)
    plt.plot(np.array(range(30))-.08,meanC[_],color = clst[0],linewidth = 4,alpha = 1  ,label = 'CCP')
    plt.plot(np.array(range(30))+.08,meanD[_],color = clst[1],linewidth = 4,alpha = 0.7,label = 'DCP')
    plt.xlim(-1,30)
    plt.ylim(1,1000)
    ax.set_yscale("log")
    plt.yticks([10e0,10e1,10e2],size = 28)
    plt.xticks(size = 28)
    plt.legend(fontsize = 22,loc='upper right')
    plt.ylabel('',fontsize = 0)
    plt.xlabel('',fontsize = 0)
